## Imports

In [1]:

from pathlib import Path
from typing import List, Dict, Any
import os
import json
import pickle
from pinecone import Pinecone, ServerlessSpec
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from bs4 import BeautifulSoup

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [27]:
## Define resume path

RESUME_DIR = Path("../resume_dir")

## Define an Index name

RESUME_INDEX_NAME = "resume-hybrid-index"

# Embeddings model

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_KEY")
TF_IDF_VECTORIZER_PATH = "tfidf_vectorizer.pkl"

## Create Pinecone Index

In [8]:
pc = Pinecone(api_key=pinecone_api_key)

if RESUME_INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=RESUME_INDEX_NAME,
        dimension=384,            # must match dense model
        metric="dotproduct", # using this because we are using hybrid search
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(RESUME_INDEX_NAME)

In [9]:
index

In [10]:
from typing import List

from langchain_core.documents import Document

 

def csr_row_to_pinecone_sparse(csr_row) -> Dict[str, List[float]]:
    coo = csr_row.tocoo()
    return {
        "indices": coo.col.tolist(),
        "values": coo.data.astype(float).tolist()
    }

 

 

# LLM for parsing resumes into structured JSON

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = gemini_api_key,
)

## Loading the resumes

In [11]:
def load_resume_text(path:Path):
    suffix = path.suffix.lower()
    if(suffix == '.pdf'):
        loader = PyPDFLoader(str(path))
        docs = loader.load()
        return "\n".join(d.page_content for d in docs)

In [12]:
#Testing 
test_path = Path("../resume_dir/Andrew_Green_Resume_27.pdf")

In [13]:
parser_text = load_resume_text(test_path)
parser_text

'ANDREW GREEN\nRecent Graduate\nContact Information:\nEmail: andrew.green@email.com\nPhone: (714) 826-3519\nLocation: Long Beach, CA\nLinkedIn: linkedin.com/in/andrewgreen\nPROFESSIONAL SUMMARY\nRecent accounting graduate with strong academic foundation and internship experience. Eager to\nbegin career in accounting with focus on financial reporting and analysis. Detail-oriented, organized,\nand committed to professional growth and development.\nTECHNICAL SKILLS\n\x7f Tax Preparation\n\x7f Financial Reporting\n\x7f Fixed Asset Management\n\x7f Payroll Processing\n\x7f Accounts Payable\n\x7f Year-End Closing\n\x7f Bank Reconciliation\n\x7f Variance Analysis\nSOFTWARE PROFICIENCY\n\x7f PowerPoint\n\x7f Microsoft Excel\n\x7f Adobe Acrobat\n\x7f SQL\n\x7f Outlook\n\x7f Sage\nPROFESSIONAL EXPERIENCE\nAccounting Intern | Premier Financial Advisors | Summer 2023\n\x7f Assisted with month-end closing procedures and financial reporting\n\x7f Processed accounts payable invoices and vendor paymen

In [14]:
def llm_parse_resume(raw_text):

    prompt = f"""

        You are a strict JSON resume parser.

        

        Return ONLY valid minified JSON. No markdown, no commentary.

        In the summary generate a summary of his experience and projects

        Schema (use exactly these keys):

        {{

          "summary": "string",

          "skills": ["string", ...],

          "CERTIFICATIONS" : ["string",...],

          "experiences": [

            {{

              "title": "string",

              "company": "string",

              "location": "string",

              "start_date": "string",

              "end_date": "string",

              "description": "string",

              "skills": ["string", ...]

            }}

          ],

          "education": [

            {{

              "degree": "string",

              "institution": "string",

              "year": "string"

            }}

          ],

          "projects": [

            {{

              "name": "string",

              "description": "string",

              "skills": ["string", ...]

            }}

          ]

        }}

        

        If something is missing, use "" or [].

        

        Resume:

        \"\"\"{raw_text[:12000]}\"\"\"

    """.strip()

    

    resp = llm.invoke(prompt)

    content = getattr(resp, "content", resp)

    return json.loads(content)

In [15]:
llm_parsed_text = llm_parse_resume(parser_text)
llm_parsed_text

{'summary': 'Recent accounting graduate with a strong academic foundation and internship experience at Premier Financial Advisors, where he assisted with month-end closing, processed accounts payable, performed bank reconciliations, maintained general ledger accounts, and created Excel spreadsheets for financial analysis. Eager to begin a career in accounting with a focus on financial reporting and analysis, he is detail-oriented, organized, and committed to professional growth.',
 'skills': ['Tax Preparation',
  'Financial Reporting',
  'Fixed Asset Management',
  'Payroll Processing',
  'Accounts Payable',
  'Year-End Closing',
  'Bank Reconciliation',
  'Variance Analysis',
  'PowerPoint',
  'Microsoft Excel',
  'Adobe Acrobat',
  'SQL',
  'Outlook',
  'Sage'],
 'CERTIFICATIONS': ['PMP', 'CGA'],
 'experiences': [{'title': 'Accounting Intern',
   'company': 'Premier Financial Advisors',
   'location': '',
   'start_date': 'Summer 2023',
   'end_date': 'Summer 2023',
   'description':

In [16]:
summary = llm_parsed_text['summary']
summary

'Recent accounting graduate with a strong academic foundation and internship experience at Premier Financial Advisors, where he assisted with month-end closing, processed accounts payable, performed bank reconciliations, maintained general ledger accounts, and created Excel spreadsheets for financial analysis. Eager to begin a career in accounting with a focus on financial reporting and analysis, he is detail-oriented, organized, and committed to professional growth.'

## LLM output required format

In [17]:
overall_text = []
def build_resume_doc(parsed_text, resume_id, file_name):
    summary = parsed_text['summary']
    skills = parsed_text['skills']
    experiences = parsed_text['experiences']
    education = parsed_text['education']
    certifications = parsed_text['CERTIFICATIONS']
    projects = parsed_text['projects']

    if (summary):
        overall_text.append(f"Summary: {summary}")
    if (skills):
        overall_text.append(f"Skills: {', '.join(skills)}")
    if (certifications):
        overall_text.append(f"Certifications: {', '.join(certifications)}")
    if (experiences):
        overall_text.append(f"Experiences: {experiences}")
    if (education):
        overall_text.append(f"Education: {education}")
    if (projects):
        overall_text.append(f"Projects: {projects}")
    
    # Metadata part
    all_skills = set()
    roles = set()
    companies = set()
    for skill in skills:
        all_skills.add(skill.lower().strip())
    for exp in experiences:
        roles.add(exp['title'].lower().strip())
        companies.add(exp['company'].lower().strip())
    
    metadata = {
        "resume_id": resume_id,
        "file_name": file_name.name,
        "skills": list(all_skills),
        "roles": list(roles),
        "companies": list(companies)
    }
    return Document(page_content='\n'.join(overall_text).strip(), metadata=metadata)

In [18]:
build_resume_doc(llm_parsed_text, 1, test_path)

Document(metadata={'resume_id': 1, 'file_name': 'Andrew_Green_Resume_27.pdf', 'skills': ['payroll processing', 'tax preparation', 'accounts payable', 'bank reconciliation', 'financial reporting', 'adobe acrobat', 'sql', 'powerpoint', 'microsoft excel', 'fixed asset management', 'sage', 'variance analysis', 'outlook', 'year-end closing'], 'roles': ['accounting intern'], 'companies': ['premier financial advisors']}, page_content="Summary: Recent accounting graduate with a strong academic foundation and internship experience at Premier Financial Advisors, where he assisted with month-end closing, processed accounts payable, performed bank reconciliations, maintained general ledger accounts, and created Excel spreadsheets for financial analysis. Eager to begin a career in accounting with a focus on financial reporting and analysis, he is detail-oriented, organized, and committed to professional growth.\nSkills: Tax Preparation, Financial Reporting, Fixed Asset Management, Payroll Processin

## Creating Docs for all the resumes in the directory

In [19]:
def create_resume_documents(path:Path):
    documents = []
    for fp in path.iterdir():
        raw_text = load_resume_text(fp)
        resume_id = fp.stem
        parsed_text = llm_parse_resume(raw_text)
        doc = build_resume_doc(parsed_text, resume_id, fp)
        documents.append(doc)
    return {
        "documents": documents
    }

In [20]:
documents = create_resume_documents(RESUME_DIR)

## Create Hybrid embeddings

In [24]:
VECTORIZER_PATH = "tfidf_vectorizer.pkl"

def create_hybrid_embeddings(documents):
    # Dense embeddings
    embed_model = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
    dense_embeddings = embed_model.embed_documents([doc.page_content for doc in documents])
    
    # Sparse embeddings
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        min_df=1,
        ngram_range=(1, 2),
    )
    corpus = [doc.page_content for doc in documents]
    tfidf_matrix = vectorizer.fit_transform(corpus)

    with open(VECTORIZER_PATH, "wb") as f:
        pickle.dump(vectorizer, f)

    print("TF-IDF vectorizer saved!")
    
    # Combine embeddings
    hybrid_embeddings = []
    for i, doc in enumerate(documents):
        sparse_emb = csr_row_to_pinecone_sparse(tfidf_matrix[i])
        hybrid_embeddings.append({
            "id": doc.metadata["resume_id"],
            "values": dense_embeddings[i],
            "sparse_values": sparse_emb,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        })
    return hybrid_embeddings

In [ ]:
#
 

# # Creating embeddings

# def step5_encode(payload):
#     docs = payload["docs"]
#     corpus = [d.page_content for d in docs]
#     embed = HuggingFaceEmbeddings(
#         model_name=EMBED_MODEL,
#         encode_kwargs={"normalize_embeddings": True},
#     )

#     dense_vectors = embed.embed_documents(corpus)

#     vectorizer = TfidfVectorizer(
#         lowercase=True,

#         stop_words="english",

#         ngram_range=(1, 2),

#         min_df=1,

#     )

#     tfidf_matrix = vectorizer.fit_transform(corpus)

 

#     #  Save vectorizer

#     with open(VECTORIZER_PATH, "wb") as f:

#         pickle.dump(vectorizer, f)

 

#     print("TF-IDF vectorizer saved!")

    

#     return {"docs": docs, "dense_vectors": dense_vectors, "tfidf_matrix":tfidf_matrix}

In [25]:
create_hybrid_embeddings(documents["documents"])[0].keys()

TF-IDF vectorizer saved!


dict_keys(['id', 'values', 'sparse_values', 'metadata', 'page_content'])

## Upsert to index

In [ ]:
def upsert_documents_to_pinecone(document: Dict[str, Any]):
    dense_vectors = document["values"]
    sparse_vectors = document["sparse_values"]
    metadata = document["metadata"]
    page_content = document["page_content"]

    if not document:
        return {
            "upserted_count": 0,
            "index": RESUME_INDEX_NAME
        }
    
    meta = {
        "text": page_content,
        **{k: v for k, v in metadata.items() if k != "text"}
    }
    document = {
        "id": document["id"],
        "values": dense_vectors,
        "sparse_values": sparse_vectors,
        "metadata": meta
    }

    index.upsert(vectors=[document])
    return {
        "upserted_count": 1,
        "index": RESUME_INDEX_NAME
    }

    

In [ ]:
for doc in create_hybrid_embeddings(documents["documents"]):
    result = upsert_documents_to_pinecone(doc)
    print(result)

{'upserted_count': 1, 'index': 'resume-hybrid-index'}
{'upserted_count': 1, 'index': 'resume-hybrid-index'}
{'upserted_count': 1, 'index': 'resume-hybrid-index'}


## Retriever Module

In [33]:
from sentence_transformers import SentenceTransformer, CrossEncoder

#Dense Model

DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
dense_model = SentenceTransformer(DENSE_MODEL_NAME)

# Reranking defines

RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

rerank_model = CrossEncoder(RERANK_MODEL_NAME)

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kshit\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet'

In [28]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = gemini_api_key,
)

In [29]:
def multi_query_ext(user_query):
    prompt = f"""
    Imagine you are helping retiver the most sutable canidates resume and you will be receving sample JD or user query about resume

    Write 3 variations based on the user question focusing different of reterival.. Do not explain what you are doing or the process

    Return only the below points
    1. The output should be sematically related to the user question
    2. Use different works compared to question but they should be related 
    3. Cover different aspects for resume filtering
    
    query_id:
    {user_query}
    """
    resp = llm.invoke(prompt)
    content = getattr(resp, "content", resp)
    return content

In [30]:
user_query = "Resume about data science"

In [31]:
mqe_query = multi_query_ext(user_query)
mqe_query

'1.  Profiles showcasing expertise in machine learning, statistical modeling, and artificial intelligence concepts.\n2.  Candidates with strong proficiencies in Python, R, SQL, and big data analytical platforms.\n3.  Resumes highlighting experience in predictive analytics, data-driven decision-making, and advanced pattern recognition projects.'

In [32]:
final_query = user_query + '\n' + multi_query_ext(user_query)
final_query

'Resume about data science\nQuery for candidates with expertise in machine learning and statistical modeling.\nSearch for individuals with proven experience in developing AI solutions and deriving data-driven insights.\nFind resumes showcasing proficiency in Python/R, big data technologies, and cloud-based analytical tools.'

In [42]:
import numpy as np

def hybrid_query(query: str, alpha: float = 0.4, top_k: int = 5):
    query = "Resume about data science"

    alpha = 0.4  # weight for dense vs sparse

    # Dense query embeddings
    embed_model = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
    query_dense_emb = embed_model.embed_query(query)
    query_dense_alpha_normalised = np.array([val * alpha for val in query_dense_emb])

    with open(TF_IDF_VECTORIZER_PATH, "rb") as f:
            vectorizer = pickle.load(f)

    # Sparse query embeddings
    q_sparse_csr = vectorizer.transform([query]).tocoo()

    if q_sparse_csr.nnz == 0:
        q_sparse = {"indices": [0], "values": [0.0]}
    else:
        q_sparse = {
                "indices": q_sparse_csr.col.tolist(),
                "values": (q_sparse_csr.data.astype(float) * alpha).tolist(),
            }

    res = index.query(
        vector=query_dense_alpha_normalised.tolist(),
        top_k=3,
        include_metadata=True,
        sparse_vector=q_sparse
    )

    out = []
    for m in res.get("matches", []):

        md = m.get("metadata", {}) or {}

        text = md.get("text", "") or ""

        preview = " ".join(text.split()[:120])  # short snippet

    

        out.append(

                {

                    "id": m["id"],                 # resume_id

                    "score": float(m["score"]),    # hybrid score

                    "preview": preview,

                    "metadata": md,

                }

        )

    return out

In [ ]:
# # Hydbrid Retrival

# def hybrid_query(query_text: str, alpha: float = 0.5, top_k: int = 8):

#     """

#     Hybrid search combining dense (1 - alpha) and sparse (alpha) scores.

 

#     alpha = 0.0 => dense-only

#     alpha = 1.0 => sparse-only (keyword)

#     """

#     alpha = float(alpha)

#     alpha = max(0.0, min(1.0, alpha))

 

#     # Load TF-IDF vectorizer trained during ingest

#     with open(TFIDF_PATH, "rb") as f:

#         vectorizer = pickle.load(f)

 

#     # Dense query embedding

#     q_dense = dense_model.encode([query_text], normalize_embeddings=True)[0]

#     q_dense = (np.asarray(q_dense, dtype=float) * (1.0 - alpha)).tolist()

 

#     # Sparse query vector

#     q_sparse_csr = vectorizer.transform([query_text]).tocoo()

#     if q_sparse_csr.nnz == 0:

#         q_sparse = {"indices": [0], "values": [0.0]}

#     else:

#         q_sparse = {

#             "indices": q_sparse_csr.col.tolist(),

#             "values": (q_sparse_csr.data.astype(float) * alpha).tolist(),

#         }

 

#     # Query Pinecone

#     res = index.query(

#         vector=q_dense,

#         sparse_vector=q_sparse,

#         top_k=top_k,

#         include_metadata=True,

#     )

 

#     out = []

#     for m in res.get("matches", []):

#         md = m.get("metadata", {}) or {}

#         text = md.get("text", "") or ""

#         preview = " ".join(text.split()[:120])  # short snippet

 

#         out.append(

#             {

#                 "id": m["id"],                 # resume_id

#                 "score": float(m["score"]),    # hybrid score

#                 "preview": preview,

#                 "metadata": md,

#             }

#         )

#     return out

In [43]:
def reranker_crossencoder(query, results):
    pairs = [(query, r['preview']) for r in results]
    scores = rerank_model.predict(pairs)
    #scores = np.mod(scores)
    rescored = []
    for r, s in zip(results,scores):
        r2 = dict(r)
        r2['rerank_cross_score'] = float(s)
        rescored.append(r2)
    return sorted(rescored, key = lambda x:x['rerank_cross_score'], reverse= True)

In [44]:
out = hybrid_query(final_query)

In [45]:
reranker_crossencoder(final_query, out)

[{'id': 'Andrew_Green_Resume_27',
  'score': 0.149416953,
  'preview': "Summary: Recent accounting graduate with a strong academic foundation and practical internship experience at Premier Financial Advisors, where he contributed to month-end closing, financial reporting, accounts payable processing, bank reconciliations, and general ledger maintenance. Eager to apply his skills in financial reporting and analysis. Skills: Tax Preparation, Financial Reporting, Fixed Asset Management, Payroll Processing, Accounts Payable, Year-End Closing, Bank Reconciliation, Variance Analysis, PowerPoint, Microsoft Excel, Adobe Acrobat, SQL, Outlook, Sage Certifications: PMP, CGA Experiences: [{'title': 'Accounting Intern', 'company': 'Premier Financial Advisors', 'location': '', 'start_date': 'Summer 2023', 'end_date': 'Summer 2023', 'description': 'Assisted with month-end closing procedures and financial reporting. Processed accounts payable invoices and vendor payments. Performed bank reconciliatio

## Generation

In [ ]:
# Prompt

# --- set the path OUTSIDE the functions ---

PROMPT_PATH = r"prompt.yaml" 
 

# --- tiny YAML loader (file only) ---

def load_prompt(path=PROMPT_PATH):

    """Load prompt config strictly from a YAML file."""

    from pathlib import Path

    import yaml  # pip install pyyaml

 

    text = Path(path).read_text(encoding="utf-8")

    cfg = yaml.safe_load(text) or {}

    

    return cfg

 

 

def render_prompt(cfg: dict, query: str, sources: str) -> str:

    vars_all = dict(cfg.get("vars", {}), query=query, sources=sources)

    return cfg["template"].format(**vars_all)

In [ ]:
# Wihtout doing any changes it will pass the input to next stage

# It will always take the input as dict
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

chain = (
RunnablePassthrough()
.assign(multi_query = RunnableLambda(lambda q:multi_query_ext(q["query"])))
.assign(results = RunnableLambda(lambda q:hybrid_query(q["multi_query"])))
.assign(rerank_results = RunnableLambda(lambda q:reranker_crossencoder(q['multi_query'] , q["results"])))
.assign(prompt_temp = RunnableLambda(lambda q:load_prompt(PROMPT_PATH)))
.assign(prompt = RunnableLambda(lambda q:render_prompt(q['prompt_temp'], q['multi_query'], q['rerank_results'])))
.assign(output = RunnableLambda(lambda q:llm.invoke(q['prompt'])))
)

user_q = "Need candidate for data science"

final_output = chain.invoke({"query": user_q})

final_output['output']

AIMessage(content='Based on the provided resumes, here\'s a ranked list of the most relevant candidates:\n\n1.  **Ashley Phillips [Ashley_Phillips_Resume_32]**\n    *   **Explanation:** Ashley is the strongest match, explicitly listing **Python** as a skill, which directly addresses the requirement for Python or R for data manipulation. Her experience as a Compliance Manager involves "leading financial reporting and analysis for a $50M+ revenue company," utilizing skills such as **Data Analysis**, **Variance Analysis**, **Trend Analysis**, **Forecasting**, and **Financial Modeling**, demonstrating capabilities in analytical methods and extracting actionable insights to drive strategic business decisions. She also mentions implementing "process improvements resulting in 20% efficiency gains," indicating a results-oriented approach from data.\n\n2.  **Angela Lewis [Angela_Lewis_Resume_09]**\n    *   **Explanation:** Angela demonstrates strong analytical capabilities through her extensive